In [1]:
# 1. 파이썬 코드에서 Matplotlib 폰트 설정
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from datetime import timedelta
from itertools import combinations
from collections import Counter

# 폰트 설정
plt.rc('font', family='Malgun Gothic')
# 마이너스 부호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

print("한글 폰트 설정이 완료되었습니다.")

# 3. 데이터 로드 및 통합 (모든 문제 풀이의 시작점)
try:
    orders_df = pd.read_csv('../data/orders.csv')
    payments_df = pd.read_csv('../data/payments.csv')
    products_df = pd.read_csv('../data/products.csv')
    shipping_df = pd.read_csv('../data/shipping.csv')
    customers_df = pd.read_csv('../data/customers.csv')

    # 모든 데이터프레임 병합
    df = pd.merge(orders_df, payments_df, on='order_id', how='left')
    df = pd.merge(df, products_df, on='product_id', how='left')
    df = pd.merge(df, customers_df, on='customer_id', how='left')
    df = pd.merge(df, shipping_df, on='order_id', how='left')

    # 데이터 전처리
    date_cols = ['order_date', 'payment_date', 'join_date', 'shipping_start_date', 'shipping_end_date']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce') # pandas의 to_datetime 검색해서 사용법 잘 봐 두고
    df['total_sales'] = df['quantity'] * df['price']
    
    print("데이터 로드 및 통합이 완료되었습니다.")

except FileNotFoundError as e:
    print(f"파일을 찾을 수 없습니다: {e}")


한글 폰트 설정이 완료되었습니다.
데이터 로드 및 통합이 완료되었습니다.


In [2]:
# 문제 5: 고객의 첫 구매부터 현재까지의 누적 구매액 변화를 고객별로 계산하세요.
# 비즈니스 목적: 고객의 성장 과정을 파악하고, VIP 고객의 성장 패턴을 이해하여 잠재적 VIP 고객을 조기에 발굴하는 데 활용합니다.
# 5명 만

In [3]:
# 출력 결과를 보고 코딩하세요
results = ''' 
고객별 누적 구매액 변화 (상위 5개 고객 샘플):
     customer_id          order_date  total_sales  cumulative_sales
1022       C0001 2024-01-03 15:03:51       290700            290700
3154       C0001 2024-06-07 00:42:31       407500            698200
3415       C0001 2024-06-24 23:09:17        28000            726200
3438       C0001 2024-06-26 07:54:05       195900            922100
6450       C0001 2025-02-05 12:35:35        51200            973300
7041       C0001 2025-03-20 12:44:59        18500            991800
7406       C0001 2025-04-15 05:26:10        29100           1020900
7357       C0002 2025-04-11 22:46:32       314000            314000
8605       C0002 2025-07-10 02:16:30       141400            455400
9703       C0002 2025-09-29 09:52:51       393200            848600
1913       C0003 2024-03-09 12:46:59       199800            199800
9461       C0003 2025-09-10 15:26:40       401000            600800
6640       C0004 2025-02-18 08:09:24        44000             44000
9444       C0004 2025-09-09 13:36:22       387600            431600
850        C0005 2023-12-21 12:02:27       132800            132800
5033       C0005 2024-10-23 14:35:32       116400            249200
9052       C0005 2025-08-12 23:21:37       378800            628000
'''

In [4]:
df.columns

Index(['order_id', 'customer_id', 'product_id', 'order_date', 'quantity',
       'payment_id', 'payment_method', 'payment_status', 'payment_date',
       'product_name', 'category', 'price', 'stock', 'name', 'gender', 'age',
       'join_date', 'city', 'shipping_id', 'shipping_company',
       'shipping_status', 'shipping_start_date', 'shipping_end_date',
       'total_sales'],
      dtype='object')

In [5]:
# 필요한 데이터 셋 
quest_cols = ['customer_id','order_date','total_sales' ]
# 생성해야할 컬럼 : cumulative_sales => 아싸 누적이닷
df_quest25 = (df[quest_cols].copy()
              .sort_values(['customer_id', 'order_date'])  # 날짜순 정렬 중요!
              .assign(cumulative_sales=lambda x: 
                      x.groupby('customer_id')['total_sales'].cumsum())
             )
# df_quest25.head()

In [6]:
# 구매 총액 상위 5명 고객 필터링
sales_top5_customers = df.groupby('customer_id')['total_sales'].sum().nlargest(5).index
df_top5 = df_quest25[df_quest25['customer_id'].isin(sales_top5_customers)]

print("고객별 누적 구매액 변화 (상위 5개 고객):")
print(df_top5)


고객별 누적 구매액 변화 (상위 5개 고객):
     customer_id          order_date  total_sales  cumulative_sales
551        C0625 2023-11-29 20:16:11        82400             82400
675        C0625 2023-12-08 03:39:23       114000            196400
1184       C0625 2024-01-15 01:49:07        66300            262700
1713       C0625 2024-02-21 23:16:47       274500            537200
2381       C0625 2024-04-11 03:45:44       272500            809700
...          ...                 ...          ...               ...
6171       C1935 2025-01-16 15:27:13       231600           1558900
6305       C1935 2025-01-25 03:48:59        28000           1586900
7662       C1935 2025-05-01 20:41:19       195600           1782500
7781       C1935 2025-05-09 08:22:36       465000           2247500
8257       C1935 2025-06-13 12:41:36       174600           2422100

[61 rows x 4 columns]


In [7]:
# 상위 5명을 그냥 처음 5명의 고객 ID 추출
top5_customers = df_quest25['customer_id'].unique()[:5]
df_top5 = df_quest25[df_quest25['customer_id'].isin(top5_customers)]

print("고객별 누적 구매액 변화 (상위 5개 고객):")
print(df_top5)

고객별 누적 구매액 변화 (상위 5개 고객):
     customer_id          order_date  total_sales  cumulative_sales
1022       C0001 2024-01-03 15:03:51       290700            290700
3154       C0001 2024-06-07 00:42:31       407500            698200
3415       C0001 2024-06-24 23:09:17        28000            726200
3438       C0001 2024-06-26 07:54:05       195900            922100
6450       C0001 2025-02-05 12:35:35        51200            973300
7041       C0001 2025-03-20 12:44:59        18500            991800
7406       C0001 2025-04-15 05:26:10        29100           1020900
7357       C0002 2025-04-11 22:46:32       314000            314000
8605       C0002 2025-07-10 02:16:30       141400            455400
9703       C0002 2025-09-29 09:52:51       393200            848600
1913       C0003 2024-03-09 12:46:59       199800            199800
9461       C0003 2025-09-10 15:26:40       401000            600800
6640       C0004 2025-02-18 08:09:24        44000             44000
9444       C0004 2025-